In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

[ RAG 구현 절차 ]
```
1.	문서의 내용을 읽는다(document_loader를 이용)
(1)	https://python.langchain.com/v0.2/docs/integrations/document_loaders/ 
(2)	https://python.langchain.com/v0.2/docs/integrations/document_loaders/microsoft_word/
%pip install --upgrade --quiet  docx2txt
2.	문서를 쪼갠다(한번에 이해하고 처리할 수 있는 입력+출력 토큰수가 제한)
(1)	 https://python.langchain.com/v0.2/docs/how_to/recursive_text_splitter/#splitting-text-from-languages-without-word-boundaries 
%pip install -qU langchain-text-splitters
3.	쪼갠 문서를 임베딩하여 vector database에 넣음
(1)	OpenAIEmbeddings나 UpstageEmbeddings이용해서 임베딩
(2)	https://python.langchain.com/v0.2/docs/integrations/vectorstores/chroma/  
%pip install –q langchain-chroma
4.	질문을 이용해 유사도 검색
5.	유사도 검색한 문서를 LLM에 질문으로 전달하여 답변 얻음(제공되는 Prompt활용)
(1)	https://python.langchain.com/v0.2/docs/tutorials/rag/
%pip install –q langchain langchainhub
http://sith.langchain.com에서 key생성 .env key(LANGCHAIN_API_KEY) 추가

```

# 2. 문서를 쪼개면서 읽기(O)

In [ ]:
import time
start = time.time()
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('./tax_docs/소득세법(법률)(제20615호)(20250701).docx')
text_spliter = RecursiveCharacterTextSplitter( #문서를 쪼개는 기준이 문자수
    chunk_size = 1500, #문서를 쪼갤 때 1500글자씩 쪼개
    chunk_overlap=200
)
# 1번째 chunk 1~1500글자
# 2번째 chunk 1250~1750글자
documents = loader.load_and_split(text_splitter=text_spliter)
runtime = time.time() - start
print('문서 쪼개면서 읽는 시간 :', runtime)

# 3. 쪼갠문서를 임베딩 -> 벡터 데이터베이스 저장
- 임베딩 모델 : upstage의 solar-embedding-1-large(기본:text-embedding-ada-002)
- 벡터 데이터베이스 : chroma

In [2]:
# https://python.langchain.com/v0.2/docs/integrations/text_embedding/upstage/
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embeddings = UpstageEmbeddings(
    model="solar-embedding-1-large"
    # model = "embedding-query"
)

In [ ]:
doc_result = embeddings.embed_documents(
    [
        "소득세법 어쩌구저쩌구",
        documents[0].page_content
    ]
)
print(len(doc_result)), print(len(doc_result[0]))

In [3]:
%%time
from langchain_chroma import Chroma
#데이터를 처음 저장할 때
# database = Chroma.from_documents(
#     documents = documents,
#     embedding = embeddings,
#     collection_name = "tax-collection", #생략시 이름 랜덤
#     persist_directory='./chroma_upstage'#생략시 로컬데이터베이스에 저장안됨. 프로그램 종료시 db날라감
# )
#이미 저장된 vector DB를 사용할 때
database = Chroma(
    embedding_function = embeddings,
    collection_name = "tax-collection",
    persist_directory='./chroma_upstage'
)

CPU times: total: 500 ms
Wall time: 743 ms


# 4. vector DB에 질문과 유사도 검색(답변 생성을 위한 retrieval)

In [6]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(query,
                                           k=3) #기본 k는 4

In [ ]:
retrieved_docs[0]

# 5. 유사도 검색으로 가져온 문서를 질문과 같이 LLM 전달하여 답변 생성

In [4]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model = "gpt-4.1-nano")

In [7]:
prompt = f"""[identity]
- 당신은 최고의 한국 소득세 전문가입니다
- [context]를 참고해서 사용자의 질문에 답변해 주세요
[context]는 다음과 같아요
{retrieved_docs}
Question : {query}"""

In [8]:
ai_message = llm.invoke(prompt)

In [9]:
print(ai_message.content)

연봉이 5,000만원인 직장인의 소득세 산출에서 적용 가능한 공제들을 고려하면 다음과 같습니다.

1. **근로소득공제**  
- 총급여액(연봉): 5,000만원  
- 근로소득공제 한도: 2,000만원  
- 계산 방법:  
  1) 5,000만원은 2,000만원 초과이므로, 공제액은 2,000만원  
  2) 따라서, 과세표준은  
  5,000만원 - 2,000만원 = 3,000만원

2. **기본 공제**  
- 기본공제는 별도로 명시되지 않았으나, 일반적으로 인적공제(본인 150만원 등)가 고려됩니다.  
- 여기서 단순 계산을 위해 과세표준: 3,000만원으로 가정합니다.

3. **소득세율 적용**  
- 2023년 소득세율(대략적 세율표 참고):  
  - 1,200만원 이하: 6%  
  - 1,200만원 초과 ~ 4,600만원 이하: 15%  

계산 방법:  
- 1,200만원까지: 1,200만원 × 6% = 72만원  
- 1,200만원 초과 3,000만원까지: (3,000만원 - 1,200만원) = 1,800만원  
  × 15% = 270만원

- 세액: 72만원 + 270만원 = 342만원

4. **추가 공제 및 세액공제**  
- 자녀세액공제, 연금계좌세액공제, 근로소득세액공제 등은 개별적 상황에 따라 달라지지만, 주어진 정보만으로는 상세히 반영하기 어렵습니다.

**결론:**  
*연봉 5,000만원인 직장인의 예상 소득세는 약 342만원 정도입니다.*

※ 참고로, 구체적인 세액 계산에는 인적공제, 연금계좌공제, 근로소득공제 등을 세분화하여 반영해야 정확한 금액 산출이 가능합니다.

혹시 더 구체적인 공제 내역이나 개인 상황 정보를 알려주시면 더욱 정밀한 계산이 가능합니다.


# 5. Augmentation을 위한 제공되는 Prompt활용하여 langchain으로 답변 생성

In [10]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"

from langchain import hub
prompt = hub.pull("rlm/rag-prompt")
prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'rlm', 'lc_hub_repo': 'rag-prompt', 'lc_hub_commit_hash': '50442af133e61576e74536c6556cefe1fac147cad032f4377b60c436e6cdcb6e'}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})])

### RetrievalQA를 통해 LLM전달 (create_retrieval_chain이 대체)
```
query -> retriever전달(백터 검색 수행) 
-> retrieval문서 -> prompt의 {context}에 삽입
-> query -> prompt의 {question}에 삽입
```

In [11]:
from langchain.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever = database.as_retriever(wearch_kwargs={'k':5}),
    chain_type_kwargs={'prompt':prompt}
)

In [12]:
ai_message = qa_chain.invoke({"query":query})

In [13]:
ai_message

{'query': '연봉 5천만원인 직장인의 소득세는 얼마인가요?',
 'result': '연봉 5천만원인 직장인의 소득세는 정확히 알 수 없습니다. 이는 소득세는 여러 공제 항목과 세율에 따라 달라지기 때문입니다. 일반적으로 근로소득공제, 자녀 세액공제, 기타 공제를 고려하면 금액이 낮아질 수 있습니다.'}